# Data Preparation Inspection

This notebook demonstrates how to inspect the data preparation logic from `run_single_super_debug`.

In [5]:
import sys
from pathlib import Path

# Add feb_exp (parent of src) and scripts so we can import src.* and data_prep_debug
feb_exp = Path("/n/home06/drooryck/codeswitching-llms/feb_exp")
sys.path.insert(0, str(feb_exp))
sys.path.insert(0, str(feb_exp / "scripts"))

from src.dataset_manager import DatasetManager
from src.model_config import ModelConfig
from data_prep_debug import prepare_data_debug

In [6]:
# Setup
prop = 0.5  # 50% French, 50% Dutch
run_id = 1
eval_prop = 0.05

data_dir = Path("/n/home06/drooryck/codeswitching-llms/feb_exp/data")
lexicon_path = Path("/n/home06/drooryck/codeswitching-llms/feb_exp/data/lexicon_sep22.json")

# Create minimal config - only required parameters needed for data preparation
# (Based on feb_exp/configs/default_model.json)
config = ModelConfig(
    # Required parameters
    n_layer=4,
    n_head=8,
    n_embd=512,
    batch_size=16,
    learning_rate=2e-4,
    warmup_steps=500,
    weight_decay=0.01,
    eval_steps=500,
    # Optional: specify training duration (one of these)
    num_train_epochs=1.0,
    # Optional: n_positions defaults to 1024 if not specified
    # vocab_size is optional and will be set from tokenizer
)

data_manager = DatasetManager(
    data_dir=str(data_dir),
    config=config,
    lexicon_path=str(lexicon_path)
)

In [7]:
# Run data preparation
train_df, val_df, test_df, train_dataloader = prepare_data_debug(
    prop=prop,
    run_id=run_id,
    eval_prop=eval_prop,
    data_manager=data_manager,
    config=config,
    batch_size=32
)

Initial train set: 591744 total | FR: 295872 | NL: 295872
Balanced train set: 591744 total | FR: 295872 | NL: 295872
Validation set: 29588 total | FR: 14794 | NL: 14794
Train pool (post-val): 562156 total | FR: 281078 | NL: 281078

Final training set:
  Total: 281078 | FR: 140539 (50.0%) | NL: 140539 (50.0%)
  Target proportion: FR=50.0%, NL=50.0%

Created DataLoader with 8784 batches
First 20 training examples (showing language interlacing):
  [  0] FR: le abeille lance le fille...
  [  1] NL: de vriend draagt de katten...
  [  2] FR: le enfant attrape le loup...
  [  3] FR: le professeur aime le aigle...
  [  4] NL: de mannen nemen de wolf...
  [  5] FR: les chevaux prennent les loups...
  [  6] NL: de beren observeren de vrouwen...
  [  7] FR: le requin casse le hibou...
  [  8] FR: les corbeaux poursuivent le ami...
  [  9] NL: de bij eet de vogels...
  [ 10] NL: de kikkers spreken de vriend...
  [ 11] FR: le ours écoute le moustique...
  [ 12] FR: les pigeons voient le souris...
 

In [8]:
# Inspect language interlacing in training set
print("Language distribution in first 100 examples:")
langs = train_df['lang'].head(100).values
print(f"FR: {(langs == 'fr').sum()}, NL: {(langs == 'nl').sum()}")
print("\nFirst 50 languages:")
print(''.join([lang.upper()[0] for lang in langs[:50]]))

Language distribution in first 100 examples:
FR: 52, NL: 48

First 50 languages:
FNFFNFNFFNNFFNNFFFFFNNNNFNNFNFNNFFFNNNNFNNFFFNFNNN


In [ ]:
# Look at consecutive language patterns
def analyze_interlacing(df, window=10):
    """Analyze how languages are interleaved."""
    langs = df['lang'].values
    
    print(f"Total examples: {len(df)}")
    print(f"FR: {(langs == 'fr').sum()}, NL: {(langs == 'nl').sum()}")
    
    # Count consecutive same-language runs
    runs = []
    current_lang = langs[0]
    run_length = 1
    
    for i in range(1, len(langs)):
        if langs[i] == current_lang:
            run_length += 1
        else:
            runs.append((current_lang, run_length))
            current_lang = langs[i]
            run_length = 1
    runs.append((current_lang, run_length))
    
    print(f"\nNumber of language runs: {len(runs)}")
    print(f"Average run length: {np.mean([r[1] for r in runs]):.2f}")
    print(f"Max run length: {max([r[1] for r in runs])}")
    
    # Show first N windows
    print(f"\nFirst {window} windows of {window} examples each:")
    for i in range(min(window, len(df) // window)):
        start = i * window
        end = start + window
        window_langs = langs[start:end]
        fr_count = (window_langs == 'fr').sum()
        nl_count = (window_langs == 'nl').sum()
        pattern = ''.join([l.upper()[0] for l in window_langs])
        print(f"  [{start:3d}-{end:3d}]: {pattern} (FR:{fr_count}, NL:{nl_count})")

analyze_interlacing(train_df)

FEB 23

In [ ]:
import pandas as pd
from pathlib import Path

# --- Set these ---
experiment = "feb20"              # or "feb20-no-plurality" orr "feb20"
ablation   = "subject"               # one of: "none", "subject", "verb", "object"
proportion = 40.0                 # e.g. 10 for 10%, 96 for 96%
run_id     = 4                   # 1..5
lang_filter = "fr"                # "fr", "nl", or None for both
n_show     = 20

# Path on netscratch
base = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/feb_exp/results")
run_dir_name = f"p{proportion:05.2f}_run{run_id:02d}"
csv_path = base / experiment / "runs" / run_dir_name / "ablation_predictions.csv"

df = pd.read_csv(csv_path)
df = df[df["ablation"] == ablation].copy()
if lang_filter is not None:
    df = df[df["language"] == lang_filter].copy()

print(f"Loaded {len(df)} rows | ablation={ablation} | lang={lang_filter or 'all'}")
df.head(n_show)

Loaded 29476 rows | ablation=object | lang=fr


,language,input,gold,prediction,ablation
3,fr,les chiens répondent de kikkers,les chiens ont répondu les grenouilles,les chiens ont répondu de répondu,object
7,fr,le chat repousse de vis,le chat a repoussé le poisson,le chat a repoussé de repoussé,object
27,fr,les lapins attendent de schaap,les lapins ont attendu le mouton,les lapins ont attendu de attendu,object
43,fr,les ânes attaquent de gans,les ânes ont attaqué le oie,les ânes ont attaqué de attaqué,object
55,fr,les poissons donnent de tijgers,les poissons ont donné les tigres,les poissons ont donné de donné,object
63,fr,les docteurs prennent de dolfijn,les docteurs ont pris le dauphin,les docteurs ont pris de pris,object
75,fr,les chèvres frappent de varken,les chèvres ont frappé le cochon,les chèvres ont frappé de frappé,object
83,fr,le docteur ouvre de haaien,le docteur a ouvert les requins,le docteur a ouvert de ouvert,object
95,fr,les chiens cassent de katten,les chiens ont cassé les chats,les chiens ont cassé de katten,object
99,fr,le poule lance de leraar,le poule a lancé le professeur,le poule a lancé de leraar,object


In [ ]:
import pandas as pd
from pathlib import Path

# --- Feb23 balanced-data runs: pick one of the 4 experiments ---
FEB23_EXPERIMENTS = {
    "v1_no_plurality": "feb23-v1-no-plurality",
    "v1_plurality_mixing": "feb23-v1-plurality-mixing",
    "v2_no_plurality": "feb23-v2-no-plurality",
    "v2_plurality_mixing": "feb23-v2-plurality-mixing",
}
experiment_version = "v2_no_plurality"   # one of: v1_no_plurality, v1_plurality_mixing, v2_no_plurality, v2_plurality_mixing
experiment = FEB23_EXPERIMENTS[experiment_version]

# --- Set these ---
ablation    = "object"    # one of: "none", "subject", "verb", "object"
proportion  = 50.0        # e.g. 10 for 10%, 96 for 96%
run_id      = 1           # 1 (feb23 runs used RUNS=(1))
lang_filter = "nl"        # "fr", "nl", or None for both
n_show      = 20

# Path on netscratch
base = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/feb_exp/results")
run_dir_name = f"p{proportion:05.2f}_run{run_id:02d}"
csv_path = base / experiment / "runs" / run_dir_name / "ablation_predictions.csv"

df = pd.read_csv(csv_path)
df = df[df["ablation"] == ablation].copy()
if lang_filter is not None:
    df = df[df["language"] == lang_filter].copy()

print(f"Loaded {len(df)} rows | {experiment_version} | ablation={ablation} | lang={lang_filter or 'all'}")
df.head(n_show)

Loaded 14779 rows | v1_plurality_mixing | ablation=object | lang=nl


,language,input,gold,prediction,ablation
3,nl,de hagedis ziet les filles,de hagedis heeft de meisjes gezien,de hagedis heeft de les filles,object
7,nl,de aap luistert le lion,de aap heeft de leeuw geluisterd,de aap heeft de le lion,object
15,nl,de muis vangt les hommes,de muis heeft de mannen gevangen,de muis heeft de les hommes,object
23,nl,de kikker geeft le pigeon,de kikker heeft de duif gegeven,de kikker hebben de le pigeon,object
27,nl,de man troost les moutons,de man heeft de schapen getroost,de man heeft de les moutons,object
43,nl,de wolf bewaart les crabes,de wolf heeft de krabben bewaard,de wolf heeft de les crabes,object
47,nl,de vlinders nemen les professeurs,de vlinders hebben de leraars genomen,de vlinders heeft de les professeurs,object
71,nl,de vogels zoeken le âne,de vogels hebben de ezel gezocht,de vogels hebben de le âne,object
75,nl,de adelaar gooit le araignée,de adelaar heeft de spin gegooid,de adelaar heeft de le araignée,object
79,nl,de haaien nemen le homme,de haaien hebben de man genomen,de haaien heeft de le homme,object
